# 17. Gradient and Optimization Analysis（spec §12）

## 学習目標
- パラメータ群別の勾配ノルム・update-to-weight 比を時系列で追う
- 損失悪化時に「どの群が不安定か」を切り分ける観点を得る

## 数式
$u_g = \dfrac{\|\Delta W_g\|_2}{\|W_g\|_2}$（群 g の1ステップ更新の相対量）。健全な AdamW では概ね $10^{-3}$ 前後。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.visualization import curves
records = read_jsonl(M2_RUN / "metrics.jsonl")
curves.update_ratio_figure(records).show()
curves.grad_norm_by_group_figure(records).show()

In [3]:
trains = [r for r in records if r["type"]=="train" and "grad_norms" in r]
last = trains[-1]["grad_norms"]
rows = [(g, last[g]) for g in sorted(last, key=lambda x:-last[x])]
print("最終step 群別勾配ノルム:")
for g, n in rows:
    print(f"  {g:12s} {n:.4f}")
clip_rate = sum(r["clip_hit"] for r in [r for r in records if r["type"]=="train"]) / len([r for r in records if r["type"]=="train"])
print(f"\nクリップ発動率 {clip_rate:.0%}")

最終step 群別勾配ノルム:
  attn_proj    0.2686
  token_emb    0.2595
  mlp          0.2265
  attn_qkv     0.1469
  pos_emb      0.1366
  norm         0.0333

クリップ発動率 5%


## Observation / Interpretation / Caveat
- **Observation**: update 比は全群で 1e-3 近傍に収まり、cosine 減衰とともに低下。token_emb と mlp の勾配ノルムが大きい。クリップは主に序盤。
- **Interpretation**: 単一 lr で全群が妥当に更新されている（群別 lr 不要）。埋め込みは全トークンの誤差を直接受けるため勾配が大きいのは自然。
- **Caveat**: この run は安定していたため「不安定時の診断」は実演できていない。M3 の LR range test（NB11）で意図的に不安定化させ、この計測器で追跡する。

**次へ（M3）**: [09_initialization_diagnostics](09_initialization_diagnostics.ipynb) 以降で校正実験に進む。